In [ ]:

# ========================================
# 필요한 라이브러리를 불러옵니다
# ========================================

import os
import time
import platform
from pprint import pprint
from typing import List, Literal, Optional


import requests
import json
import pandas as pd
import numpy as np
import time

import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from pydantic_ai import Agent, ModelRetry
from pydantic_ai.models.google import GoogleModelSettings
from concurrent.futures import ThreadPoolExecutor, as_completed

# 운영체제별 한글 폰트 설정
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # macOS
    plt.rcParams['font.family'] = 'AppleGothic'
else:  # Linux
    plt.rcParams['font.family'] = 'NanumGothic'

# 마이너스 기호 깨짐 방지
plt.rcParams['axes.unicode_minus'] = False

# 출력 짤림 방지
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 50)

# .env 파일에서 API 키 로드
load_dotenv()
api_key = os.getenv('GEMINI_API_KEY')
gemini_model = os.getenv('GEMINI_MODEL', 'gemini-3.1-flash-lite-preview')
youtube_api_key = os.getenv('YOUTUBE_API_KEY')
youtube_api_key2 = os.getenv('YOUTUBE_API_KEY2')
youtube_api_key3 = os.getenv('YOUTUBE_API_KEY3')

# PydanticAI는 GEMINI_API_KEY 환경변수를 자동으로 인식합니다
# 모델 ID 형식: 'google-gla:{모델명}'
model_id = f'google-gla:{gemini_model}'

# API 키 유효성 검사
api_key_valid = api_key and 'YOUR_API_KEY' not in api_key
print(f"API 키 설정 확인: {'O' if api_key_valid else 'X'}")
if not api_key_valid:
    print("[주의] .env 파일에서 GEMINI_API_KEY를 실제 API 키로 설정해주세요!")
print(f"모델 확인: {model_id}")

youtube_api_key_valid = youtube_api_key and 'YOUR_API_KEY' not in youtube_api_key
print(f"유튜브 API 키 설정 확인: {'O' if youtube_api_key_valid else 'X'}")
if not api_key_valid:
    print("[주의] .env 파일에서 YOUTUBE_API_KEY를 실제 API 키로 설정해주세요!")

# API 호출 간격 (초) - 무료 플랜에 맞게 조정
API_DELAY = 3

API 키 설정 확인: O
모델 확인: google-gla:gemini-3.1-flash-lite-preview
유튜브 API 키 설정 확인: O


In [2]:

# =============================================================
# IT 기업리스트
# 생성된 데이터 프레임: it_channel_list
# 생성된 csv 파일 이름: IT채널정보조회.csv
# =============================================================
with open('./data/checkpoint_IT(배치처리).json', encoding='utf-8') as f:
    it_data = json.load(f) # Ai Agent가 정리한 IT기업의 채널 리스트 로드

it_records = [] # 데이터를 담을 빈 리스트 생성

# channels에 들어있는 dict 확장
for company in it_data:
    company_name = company['company_name']
    has_official = company['has_official']
    
    for ch in company['channels']:
        it_records.append({
            'company_name': company_name,
            'has_official': has_official,
            'channel_id': ch.get('channel_id'),
            'channel_name': ch.get('channel_name'),
            'channel_handle': ch.get('channel_handle'),
            'channel_url': ch.get('channel_url'),
            'channel_reason': ch.get('channel_reason')
        })

it_channel_list = pd.DataFrame(it_records)
it_channel_list.to_csv('./data/IT채널정보조회.csv')
it_channel_list.head()

# =============================================================
# 작성된 기업 리스트를 기반으로 수기로 하나하나 검증 중 . . .
# 완료!
# ============================================================= 

# 기업 채널 리스트 가져오기
it_channel_list2 = pd.read_csv('./data/IT리스트 정리.csv', encoding='euc-kr')

# 채널이 없는 기업 제외
it_channel_list2 = it_channel_list2.loc[it_channel_list2['channel_name'].notna(), ['company_name', 'channel_name', 'channel_id', 'channel_handle']]

# 정리된 채널 개수 확인
print('정리된 기업 채널 개수 :' , len(it_channel_list2['company_name']), '개')


정리된 기업 채널 개수 : 49 개


In [3]:

# =============================================================
# FOOD 기업리스트
# 생성된 데이터 프레임: food_channel_list
# 생성된 csv 파일 이름: FOOD채널정보조회.csv
# =============================================================

with open('./data/checkpoint_FOOD(배치처리).json', encoding='utf-8') as f:
    food_data = json.load(f) # Ai Agent가 정리한 FOOD기업의 채널 리스트 로드

food_records = [] # 데이터를 담을 빈 리스트 생성

# channels에 들어있는 dict 확장
for company in food_data:
    company_name = company['company_name']
    has_official = company['has_official']
    
    for ch in company['channels']:
        food_records.append({
            'company_name': company_name,
            'has_official': has_official,
            'channel_id': ch.get('channel_id'),
            'channel_name': ch.get('channel_name'),
            'channel_handle': ch.get('channel_handle'),
            'channel_url': ch.get('channel_url'),
            'channel_reason': ch.get('channel_reason')
        })

food_channel_list = pd.DataFrame(food_records)
food_channel_list.to_csv('./data/FOOD채널정보조회.csv')
food_channel_list.head()

# =============================================================
# 작성된 기업 리스트를 기반으로 수기로 하나하나 검증 중 . . .
# 완료!
# ============================================================= 

# 기업 채널리스트 가져오기
food_channel_list2 = pd.read_csv('./data/Book(실제 검증 채널).csv', encoding='utf-8-sig')

# 채널이 없는 기업 제외
food_channel_list2 = food_channel_list2.loc[food_channel_list2['channel_name'].notna(), ['company_name', 'channel_name', 'channel_id', 'channel_handle']]

# 정리된 채널 개수 확인
print('정리된 기업 채널 개수', len(food_channel_list2['company_name']), '개')


정리된 기업 채널 개수 70 개


---
#### 채널 정보 수집
   수집 목적:
   1. 채널 정보를 분석하여 사용할 최종 채널 리스트를 선정하기 위함.
   2. 이후 영상정보 검색시 채널 id 정보가 필요.

---


In [ ]:
# =============================================================
# 남은 토큰 관리를 위한 남은 토큰 수 입력
#   - https://console.cloud.google.com/iam-admin/quotas?hl=ko
#   - youtube api 키와 연결된 프로젝트 선택
#   - 서비스 Youtube Data API v3의 현재 사용량 확인.
# =============================================================
print('API 사용량 확인: https://console.cloud.google.com/iam-admin/quotas?hl=ko')
total_unit = 10000 - int(input('이미 사용하신 사용량을 입력해주세요. 예) 256 :'))
print('현재 남은 유닛 수 :', total_unit)

In [33]:

# =============================================================
# 채널 정보 불러오기 API : get_channel_info(arg1, arg2, arg3, arg4)
# arg1: 채널 정보를 불러올 때 사용할 유튜브 API키 입니다. (필수)
# arg2: 정보를 불러올 채널의 핸들명 입니다. (arg2, arg3 중 하나만 선택 입력)
# arg3: 정보를 불러올 채널의 핸들명 입니다. (arg2, arg3 중 하나만 선택 입력)
# arg4: 채널의 정보를 담을 리스트 변수명 입니다. 함수가 동작되기 전에 미리 생성되어 있어야 합니다.
# 
# [불러올 정보]
# 1. 채널 아이디(channel_id): 채널이 가지고 있는 고유 아이디 값입니다.
#   - 이후에 유튜브 검색 API 함수에서 쓰입니다. (채널 별 영상 아이디를 가져올 거임)
# 2. 채널 이름(title): 불러올 채널의 이름입니다.
# 3. 채널 설명(description): 채널에 대한 설명입니다.
# 4. 구독자 수(subscriber_count): 채널의 구독자 수입니다.
# 5. 총 조회수(view_count): 채널 모든 영상의 조회수를 합한 값입니다.
# 6. 총 영상수(video_count): 채널이 업로드한 공개된 영상의 개수입니다. (비공개 영상 포함 X)
# 7. 채널 썸네일(thumbnails): 채널의 썸네일입니다. url형식으로 제공됩니다.
# 8. 채널 개설일(created_date): 채널의 개설 날짜입니다.
# 
# 사용할 API: 유튜브 채널 API
#   - 비용: 채널 1개 당 1 유닛
#
# ※ 주의사항
#   - 이 함수는 handle의 리스트나, channel_id의 리스트를 받아서 한꺼번에 출력할 수 없습니다.
#   - 따라서 handle, channel_id는 함수에 한개씩만 들어가야 합니다. (리스트 넣지 말라는 얘기입니다.)
#   - 여러 개의 기업 채널에 대해서 정보를 얻고 싶다면, 반복문 속에 함수를 넣으십시오.
#   - 이 함수는 사용한 유닛량을 알려주지 않습니다. 정보를 얻고자 하는 채널의 수와 유닛량이 같다는 점을 참고해주십시오.
# =============================================================

def get_channel_info(api_key, handle=None, channel_id=None, list_name=None):
    url = 'https://www.googleapis.com/youtube/v3/channels'
    
    params = {
        'part': 'snippet,statistics,brandingSettings',
        'key':api_key
    }
    
    if handle:
        params['forHandle'] = handle
    elif channel_id:
        params['id'] = channel_id
    else:
        raise ValueError('handle 또는 channel_id 중 반드시 하나는 입력해야 합니다.')
    
    if list_name == None:
        raise ValueError('채널 정보를 받을 리스트를 입력해주세요')
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    data = r.json()
    
    if not data.get('items'):
        return None
    ch = data['items'][0]
    temp ={
        'channel_id': ch['id'], # 채널 아이디
        'title': ch['snippet']['title'], #채널 이름
        'description': ch['snippet'].get('description'), #채널 설명
        'subscriber_count': ch.get('statistics', {}).get('subscriberCount'), # 구독자 수
        'view_count': ch.get('statistics',{}).get('viewCount'), # 채널 총 조회수
        'video_count': ch.get('statistics',{}).get('videoCount'), #채널 총 영상 수
        'thumbnails': ch['snippet'].get('thumbnails').get('default').get('url'), # 채널 프로필 사진?
        'created_date': ch['snippet'].get('publishedAt') # 채널 생성 날짜
    }
    list_name.append(temp)
    return list_name


In [32]:
# =============================================================
# IT 기업의 유튜브 채널 정보 수집 시작
# =============================================================
it_handle_list = it_channel_list2['channel_handle'].to_list() # 정제된 IT 채널 리스트를 가져옴

print('='*50)
print('현재 남은 유닛 수 :', total_unit)
print('사용 예정 유튜브 API 유닛 :', len(it_handle_list), 'units')

it_list = [] # it 채널의 정보를 담을 리스트 생성
unit= 0 # 유닛 수 계산 시작!
for handle in it_handle_list:
    get_channel_info(youtube_api_key, handle=handle, list_name=it_list)
    unit += 1
total_unit = total_unit - unit
print()
print('실제 사용된 유튜브 API 유닛: ', unit, 'units')
print('남은 유닛 수: ', total_unit)
print('='*50)

# IT 채널 정보 리스트 데이터프레임으로 변환
it_channel = pd.DataFrame(it_list)
print()
print('수집 결과:')
display(it_channel)
# csv로 저장
it_channel.to_csv('./data/IT채널정보조회.csv', encoding='utf-8-sig')



현재 남은 유닛 수 : 7200
사용 예정 유튜브 API 유닛 : 49 units
남은 유튜브 유닛:  7200
남은 유튜브 유닛:  7200
남은 유튜브 유닛:  7200
남은 유튜브 유닛:  7200
남은 유튜브 유닛:  7200
남은 유튜브 유닛:  7200
남은 유튜브 유닛:  7200
남은 유튜브 유닛:  7200
남은 유튜브 유닛:  7200
남은 유튜브 유닛:  7200
남은 유튜브 유닛:  7200
남은 유튜브 유닛:  7200
남은 유튜브 유닛:  7200
남은 유튜브 유닛:  7200
남은 유튜브 유닛:  7200
남은 유튜브 유닛:  7200
남은 유튜브 유닛:  7200
남은 유튜브 유닛:  7200
남은 유튜브 유닛:  7200
남은 유튜브 유닛:  7200
남은 유튜브 유닛:  7200
남은 유튜브 유닛:  7200
남은 유튜브 유닛:  7200
남은 유튜브 유닛:  7200
남은 유튜브 유닛:  7200
남은 유튜브 유닛:  7200
남은 유튜브 유닛:  7200
남은 유튜브 유닛:  7200
남은 유튜브 유닛:  7200
남은 유튜브 유닛:  7200
남은 유튜브 유닛:  7200
남은 유튜브 유닛:  7200
남은 유튜브 유닛:  7200
남은 유튜브 유닛:  7200
남은 유튜브 유닛:  7200
남은 유튜브 유닛:  7200
남은 유튜브 유닛:  7200
남은 유튜브 유닛:  7200
남은 유튜브 유닛:  7200
남은 유튜브 유닛:  7200
남은 유튜브 유닛:  7200
남은 유튜브 유닛:  7200
남은 유튜브 유닛:  7200
남은 유튜브 유닛:  7200
남은 유튜브 유닛:  7200
남은 유튜브 유닛:  7200
남은 유튜브 유닛:  7200
남은 유튜브 유닛:  7200
남은 유튜브 유닛:  7200

실제 사용된 유튜브 API 유닛:  49 units
남은 유닛 수:  7151

수집 결과:


,channel_id,title,description,subscriber_count,view_count,video_count,thumbnails,created_date
0,UCA5vKQkvC9zlTDOSNtoMdCQ,SK telecom,SK telecom 공식 YouTube 채널,1130000,1313042518,2705,https://yt3.ggpht.com/ytc/AIdro_m_9Gd-5ejb35YE...,2011-01-12T06:00:00Z
1,UCqi8fAllvDe9-QCDlhxSg8Q,SK AI SUMMIT 2025,"AI NOW & NEXT\n오늘날 AI로 이룬 혁신을 경험하고, 내일의 도약을 위한...",5320,192389,324,https://yt3.ggpht.com/BUovhY65Fkq3930DuKeBjwJB...,2022-08-04T08:01:36.337173Z
2,UCp3BSINVC7Ggj4kChEUiy7Q,LG유플러스 (LG Uplus),LG U+ 공식 Youtube 채널 \n,660000,1942109825,3713,https://yt3.ggpht.com/MXGutyn0ZJqNgLb5bPGj514z...,2011-12-19T07:47:10Z
3,UCVaIGU8ch9zG42j-rAG6VoA,삼성SDS,AI 풀스택 역량으로 기업의 AX를 선도하는 업무 혁신 파트너\n삼성SDS 공식 Y...,43200,74705486,1137,https://yt3.ggpht.com/v7Jgiy1VlR5ps-jDUHkpnzgf...,2016-10-04T07:31:53Z
4,UCqz8F0bdoDiyzgdYPZAdA-Q,LG CNS,Global AX 전문기업 LG CNS의 공식 유튜브 채널입니다.,76200,68795597,473,https://yt3.ggpht.com/gm9Yi3503swI8kz0XgMxY-xZ...,2012-05-16T04:55:12Z
5,UCwR9k7QggFEfeHkvTR31JcA,SK브로드밴드_B tv,SK 브로드밴드 공식 유튜브 채널입니다\n,75600,179249506,4330,https://yt3.ggpht.com/ojNY0Wvjfp63Lazv7n4DDAOo...,2014-06-20T06:54:37Z
6,UCjyYouHWnID_L4QaQ6U4voQ,네이버 NAVER,네이버 공식 유튜브 채널입니다.,32600,288223085,862,https://yt3.ggpht.com/D-DsROA_yjJi-B1x4nnHXXws...,2012-11-22T12:07:43Z
7,UCBjvBJgIp3NGkrTBEfWBUVw,카카오,일상에 스며든 카카오의 다양한 이야기를 전합니다. 카카오 공식 유튜브 채널.,132000,254319875,910,https://yt3.ggpht.com/SSvyEl7-5J4DlWtbrHfeERIs...,2012-03-26T03:51:31Z
8,UCRBtxwx23bzxyr5R9vFRYhw,현대오토에버,여기는 현대오토에버 공식 YouTube 채널입니다.,10200,3748078,147,https://yt3.ggpht.com/L2QnWR1xkuWsogbZ91hIx6CY...,2019-10-29T05:34:14.878126Z
9,UCqqoiebjYnMqfhx-4f1awKg,Microsoft Korea,Microsoft 목표와 가치는 전세계의 사람과 기업이 잠재력을 최대한 발휘할 수 ...,18600,71104335,1845,https://yt3.ggpht.com/MRNhR0pURHfaJW1zLqGwGc7-...,2011-03-18T07:38:13Z


In [7]:
# =============================================================
# 식품 기업의 유튜브 채널 정보 수집 시작
# =============================================================
food_handle_list = food_channel_list2['channel_handle'].to_list() # 정제된 IT 채널 리스트를 가져옴

print('='*50)
print('현재 남은 유닛 수 :', total_unit)
print('사용 예정 유튜브 API 유닛 :', len(food_handle_list), 'units')

food_list = [] # it 채널의 정보를 담을 리스트 생성
unit= 0 # 유닛 수 계산 시작!
for handle in food_handle_list:
    get_channel_info(youtube_api_key, handle=handle, list_name=food_list)
    unit += 1
total_unit = total_unit - unit
print()
print('실제 사용된 유튜브 API 유닛: ', unit, 'units')
print('남은 유닛 수: ', total_unit)
print('='*50)

# IT 채널 정보 리스트 데이터프레임으로 변환
food_channel = pd.DataFrame(food_list)
print()
print('수집 결과:')
display(food_channel)
# csv로 저장
food_channel.to_csv('./data/FOOD채널정보조회.csv', encoding='utf-8-sig')

현재 남은 유닛 수 : 9751
사용 예정 유튜브 API 유닛 : 70 units

실제 사용된 유튜브 API 유닛:  70 units
남은 유닛 수:  9681

수집 결과:


,channel_id,title,description,subscriber_count,view_count,video_count,thumbnails,created_date
0,UCsM07dUwo0WOWhFbVsG8K6A,CU [씨유튜브],"편의점의 새로운 정의!\n트렌드의 중심, 재미의 기준을 만들지😎✌️""\n매일 새로운...",927000,305247518,1173,https://yt3.ggpht.com/ytc/AIdro_mI4fk2-pjTlfnJ...,2012-06-05T11:33:04Z
1,UCnL1m_iPeN8vX7zFHoB4SYg,테헤란로405,테헤란로405에서 좋은 친구 BGF의 주요 소식을 살펴보세요!\n-\n편의점 CU를...,2120,986927,501,https://yt3.ggpht.com/XgW8KshcFJNgLiN92DYhQ3KQ...,2017-03-03T10:34:35Z
2,UCl0HemfAt7dShJRf59_cuGQ,GS25 l 이리오너라,GS25 공식 유튜브 채널📻\n,1060000,222994020,2580,https://yt3.ggpht.com/AajkhhUfGCTlIBM8YvvfZmWK...,2012-12-26T06:54:54Z
3,UChD4kH8BcI49njXGL5-makg,GS리테일 뉴스룸,GS리테일 뉴스룸 공식 유튜브 채널입니다.\n최신 뉴스와 다양한 소식들을 영상으로 ...,2950,1043786,167,https://yt3.ggpht.com/dRpIX9kTKHJJXBJwtWoyXtBB...,2015-05-18T08:16:28Z
4,UCG_gfzGVpsxGwYuFgH8-tNA,대상 DAESANG,더 많은 것들을 존중의 대상으로 DAESANG\n#Respect for more #...,18500,330712382,254,https://yt3.ggpht.com/UGJpLrZkEdbCDTFulkdMEeaH...,2015-12-10T08:58:42Z
5,UCfv1dkNKCFUtZouvp3DhS3g,대상그룹 DAESANG 디튜브,대상그룹(DAESANG GROUP) 공식 소통 채널\n‘대상주식회사’의 식품 이야기...,10800,5755529,277,https://yt3.ggpht.com/JNGLkJj-SitafANGlfbseU-j...,2020-09-04T01:25:26.250343Z
6,UCDryA2PIGnbs9NYn_63rISw,CJ제일제당,CJ제일제당 공식 유튜브 채널입니다. \n,491000,34922991,204,https://yt3.ggpht.com/ULAliXxe7B5_ysZS8vIsPs18...,2011-01-26T00:52:39Z
7,UCOV_BpGzvmH7ZtjQOWg0bSQ,제당슈만,⭐제당슈만⭐ CJ제일제당이 당신을 🔥슈퍼스타🔥로 만들어 드립니다!\nCJ제일제당의 ...,20900,6641901,216,https://yt3.ggpht.com/Yk3vnP2gHzidyACXSqtTo1LU...,2020-05-15T11:50:40.345514Z
8,UCGmLtNZFTL6PiKsz3Vpdgbw,비비고 (bibigo),bibigo Official YouTube Channel\n\n,88100,275870330,307,https://yt3.ggpht.com/nQgCSi2mTp99T8VaSEH5MVyZ...,2013-04-02T11:37:09Z
9,UCFXwXfbr4b1HHl-YKCOaydA,맛깔스튜디오 by롯데웰푸드,맛있는 거 말고 재밌는 거도 기깔나게 잘합니다.\n롯데웰푸드의 맛깔나는 컨텐츠 제작...,147000,490800635,1311,https://yt3.ggpht.com/LJScwH9HDweKeTnwo8UAMcVd...,2011-11-23T07:54:50Z


---
#### 채널 선정기준
    - 채널 정보 데이터 EDA를 통해 선정 기준 정립

---

In [ ]:
# =======================================================================
# 1차 채널 선정 기준 정하기
# =======================================================================
# 
# channel_list(arg1, arg2)
# 
# arg1: 데이터프레임이름
# arg2: 저장할 경로 (예: './data/test.csv')
# 
# 1. 구독자 1만 명 미만 채널 제거 : 마이크로 인플루언서 최소 기준
# 2. 채널 나이 3년 미만 채널 제거 : 계절성을 반영하기 위한 기준
#
# =======================================================================

def channel_list(ch_data, save_path=None):
    
    # 숫자형 컬럼 정수로 변환
    ch_data['subscriber_count'] = ch_data['subscriber_count'].astype(int)
    ch_data['video_count'] = ch_data['video_count'].astype(int)
    ch_data['view_count'] = ch_data['view_count'].astype(int)
    
    # 1. 구독자 1만 명 미만 채널 제거
    ch_data = ch_data[ch_data['subscriber_count']>= 10000] 
    
    # 2. 채널 나이 3년 미만 채널 제거
    ch_data['year'] = pd.to_datetime(ch_data['created_date'], errors='coerce', format='mixed', utc=True).dt.year
    ch_data['channel_age'] = 2026 - ch_data['year']
    ch_data = ch_data[ch_data['channel_age'] >= 3]
    
    # # 3. 연 평균 4회 미만 영상 업로드 채널 제거 -> IT는 이 조건 있으나 없으나 동일. F&B는 이 조건 추가시 폴바셋 채널 삭제 https://www.youtube.com/channel/UCabcfEPMYK0rvbGIl3Gr3Ag
    # videos_thr = ch_data['channel_age'] * 4
    # ch_data = ch_data[ch_data['video_count'] >= videos_thr]
    
    # CSV 파일로 저장
    ch_data.to_csv(save_path, index=False, encoding='utf-8-sig')
    return ch_data




In [ ]:
# =======================================================================
# 1차 채널 선정 : IT 채널 리스트 : it_filtered_channel_list
# =======================================================================
it_filtered_channel_list = channel_list(it_channel, './data/IT_Filtered_Channel_list.csv')
len(it_filtered_channel_list)

In [ ]:
# =======================================================================
# 1차 채널 선정 : FOOD 채널 리스트 : food_filtered_channel_list
# =======================================================================
food_filtered_channel_list = channel_list(food_channel, './data/FOOD_Filtered_Channel_list.csv')
len(food_filtered_channel_list)

---
#### 채널 영상정보 검색
    - 채널이 가지고 있는 영상의 Video_id를 수집
    - 이후에 진행할 영상 상세정보 불러오기에 사용
---

In [ ]:
# =======================================================================
# 채널의 uploads playlist를 이용해 영상 정보를 수집 : get_playlist_video_info(arg1, arg2, arg3, arg4)
#
# arg1: 유튜브 API 키 (필수)
# arg2: 유튜브 채널 ID (필수)
# arg3: 가져올 최대 영상 개수 (기본값: 50)
# arg4: 저장할 csv 경로 (필수)
#
# 비용: 1회 당 2유닛
#   channels: 
#
# [수집할 정보]
# - channel_id: 채널의 고유 아이디 값
# - video_id: 영상의 고유 아이디 값
# - published_at: 
# - title
# - description
# =======================================================================

def get_playlist_video_info(api_key, channel_id=None, max_results=50, save_path=None):
    
    if save_path is None:
        raise ValueError("저장할 경로를 입력해주세요. 예: save_path='./data/youtube_data.csv'")

    if not channel_id:
        raise ValueError("channel_id를 입력해주세요")

    # 기존 CSV에 저장된 video_id 불러오기
    existing_video_ids = set()

    if os.path.exists(save_path):
        existing_df = pd.read_csv(save_path)
        if 'video_id' in existing_df.columns:
            existing_video_ids = set(existing_df['video_id'].dropna().astype(str))

    # 채널의 uploads playlist ID 가져오기
    url = 'https://www.googleapis.com/youtube/v3/channels'
    channel_params = {
        'part': 'contentDetails',
        'id': channel_id,
        'key': api_key
    }

    r = requests.get(url, params=channel_params, timeout=30)
    print(r.url)

    if not r.ok:
        print('status_code:', r.status_code)
        try:
            print('error_json:', r.json())
        except Exception:
            print('error_text:', r.text)
        return []

    channel_data = r.json()
    items = channel_data.get('items', [])

    if not items:
        print('채널 정보를 찾을 수 없습니다.')
        return []
    
    # 업로드된 플레이리스트 아이디 가져오기 
    uploads_playlist_id = (
        items[0]
        .get('contentDetails', {})
        .get('relatedPlaylists', {})
        .get('uploads')
    )

    if not uploads_playlist_id:
        print('uploads playlist ID를 찾을 수 없습니다.')
        return []
    
    # playlistItems로 영상 정보 수집
    playlist_url = 'https://www.googleapis.com/youtube/v3/playlistItems'
    all_items = []
    next_page_token = None

    while len(all_items) < max_results:
        remain = max_results - len(all_items)
        request_count = min(50, remain)

        playlist_params = {
            'part': 'snippet',
            'playlistId': uploads_playlist_id,
            'maxResults': request_count,
            'key': api_key
        }

        if next_page_token:
            playlist_params['pageToken'] = next_page_token

        r = requests.get(playlist_url, params=playlist_params, timeout=30)
        print(r.url)

        if not r.ok:
            print('status_code:', r.status_code)
            try:
                print('error_json:', r.json())
            except Exception:
                print('error_text:', r.text)
            break

        data = r.json()
        items = data.get('items', [])

        if not items:
            break

        for video in items:
            snippet = video.get('snippet', {})
            resource_id = snippet.get('resourceId', {})
            video_id = resource_id.get('videoId')

            if not video_id:
                continue

            video_id = str(video_id)

            # 중복이면 건너뛰기만 함 (중단 X)
            if video_id in existing_video_ids:
                continue

            all_items.append({
                'channel_id': snippet.get('channelId'),
                'video_id': video_id,
                'published_at': snippet.get('publishedAt'),
                'title': snippet.get('title'),
                'description': snippet.get('description')
            })

            # 중복 방지용 set에도 즉시 추가
            existing_video_ids.add(video_id)

            if len(all_items) >= max_results:
                break

        next_page_token = data.get('nextPageToken')
        if not next_page_token:
            break


    # 4. CSV 저장

    if all_items:
        new_df = pd.DataFrame(all_items)
    
        if not os.path.exists(save_path):
            new_df.to_csv(save_path, index=False, encoding='utf-8-sig')
        else:
            new_df.to_csv(save_path, mode='a', header=False, index=False, encoding='utf-8-sig')
    
    return all_items

In [ ]:
# =======================================================================
# 유닛비용 미리 계산
# =======================================================================

print('가지고 있는 유닛 수: ', total_unit)
total_video_count = np.sum(it_filtered_channel_list['video_count'])
total_spend_unit = int(np.ceil(total_video_count / 50))*2
print('총 영상 수 :', total_video_count)
print('예상 유닛 비용: ',total_spend_unit)

In [ ]:
# =======================================================================
# IT 기업 채널의 영상정보 수집 API
# =======================================================================

print('가지고 있는 유닛 수: ', total_unit)

for channel_id in it_filtered_channel_list['channel_id'].to_list()[2:]:
    
    video_count = it_filtered_channel_list.loc[
        it_filtered_channel_list['channel_id'] == channel_id,
        'video_count'
    ].iloc[0]
    
    get_playlist_video_info(
        youtube_api_key,
        channel_id=channel_id,
        max_results=video_count,
        save_path='./data/IT_video_ids.csv'
    )
    
    print(f"{channel_id} 채널의 영상 수 : {video_count}")
    
    spend_unit = int(np.ceil(video_count / 50))*2
    total_unit -= spend_unit
    
    print('현재 남은 유닛 수 :', total_unit)




In [ ]:
# =======================================================================
# 유닛 비용 계산
# =======================================================================

print('가지고 있는 유닛 수: ', total_unit)
total_video_count = np.sum(food_filtered_channel_list['video_count'])
total_spend_unit = int(np.ceil(total_video_count / 50))*2
print('총 영상 수 :', total_video_count)
print('예상 유닛 비용: ',total_spend_unit)

In [ ]:
# =======================================================================
# FOOD 기업 채널의 영상정보 수집 API
# =======================================================================

print('가지고 있는 유닛 수: ', total_unit)

for channel_id in food_filtered_channel_list['channel_id'].to_list():
    
    video_count = food_filtered_channel_list.loc[
        food_filtered_channel_list['channel_id'] == channel_id,
        'video_count'
    ].iloc[0]
    
    get_playlist_video_info(
        youtube_api_key,
        channel_id=channel_id,
        max_results=video_count,
        save_path='./data/FOOD_video_ids.csv'
    )
    
    print(f"{channel_id} 채널의 영상 수 : {video_count}")
    
    spend_unit = int(np.ceil(video_count / 50))*2
    total_unit -= spend_unit
    
    print('현재 남은 유닛 수 :', total_unit)





---
#### 영상 상세 정보 출력
    - 이제 본 분석에 사용될 영상의 상세 정보를 가져온다.

---

In [ ]:
# =======================================================================
# 영상 상세 정보 불러오기 : get_video_info(arg1, arg2, arg3)
# arg1: 영상 상세정보를 가져올 유튜브 API키 입니다. (필수)
# arg2: 영상 상세정보를 가져올 Video_id의 리스트입니다. (필수)
# arg3: 영상 상세정보를 저장할 경로입니다. (필수. 예: save_path = './data/test.csv')
#
# [불러올 정보]
# 1. 영상 아이디(video_id): 영상이 가지고 있는 고유 아이디 값입니다.
# 2. 영상 제목(title): 영상 제목
# 3. 채널 아이디(channel_id): 해당 영상을 업로드한 채널의 고유 아이디 값입니다.
# 4. 채널 이름(channel_title): 해당 영상을 업로드한 채널의 이름입니다.
# 5. 영상 설명(description): 업로더가 작성한 해당 영상의 설명입니다.
# 6. 영상 업로드 날짜(upload_date): 영상이 업로드된 날짜입니다.
# 7. 영상 키워드 태그 목록(tags): 영상의 키워드 태그 목록입니다.
# 8. 영상 카테고리 아이디(category_id): 영상이 속해있는 카테고리의 고유값 입니다.
# 9. 조회수(view_count): 해당 영상의 조회수입니다.
# 10. 좋아요 수(like_count): 해당 영상의 좋아요 수 입니다.
# 11. 댓글 수(like_count): 해당 영상의 댓글 수 입니다.
# 12. 영상 길이(duration): 해당 영상의 길이입니다. (형식: PT#H#M#S, T: 일, H: 시간, M: 분, S: 초)
# 13. 고화질 여부(definition): 영상을 고화질로 시청할 수 있는지 여부입니다. (hd(High Definition): 고화질 시청 가능, sd(Standard Definition): 일반화질 시청)
# 14. 동영상 라이선스(license): 동영상 라이선스 정보입니다. creativeCommon: 출처를 표시하면 영상의 재편집/재배포가 가능, youtube: 영상 재편집/재배포 불가능
# 15. 임베딩 가능 여부(emabeddable): 영상을 다른 웹사이트에 퍼갈 수 있는지 여부 (광고 영상을 전광판이나 특정 홈페이지에서 계속 송출한다면 조회수가 올라가지 않을까?)
# 16. 유료 PPL 여부(has_paid_product_placement): 유료 PPL 사용 여부 (True/False)
# =======================================================================

def get_video_info(api_key, video_ids=None, save_path=None):
    
    # 영상 정보 수집 API url 가져오기
    url = 'https://www.googleapis.com/youtube/v3/videos'
    all_results = []
    
    if not video_ids:
        print('video_ids가 없습니다.')
        return pd.DataFrame()
    
    if save_path == None:
        raise ValueError('파일을 저장할 경로를 입력해주세요.')
    
    # 기존 video_id 불러오기
    existing_video_ids = set()
    
    if save_path and os.path.exists(save_path):
        existing_df = pd.read_csv(save_path)
        if 'video_id' in existing_df.columns:
            existing_video_ids = set(existing_df['video_id'].dropna().astype(str)) # video_id 결측없이/문자형/set(중복 없음, 순서없음, 집합개념) 형태로 가져오기
    
    # 중복 제거 & 리스트로 변환
    video_ids = [str(v) for v in video_ids if v] # 만약 영상 아이디가 video_ids에 포함되고, 값이 존재한다면 문자열로 변환된 영상 아이디를 출력 반복
    video_ids = list(dict.fromkeys(video_ids))
    
    # 이미 저장된 video_id 제외
    if existing_video_ids:
        video_ids = [vid for vid in video_ids if vid not in existing_video_ids]
    
    # 불러올 video_id가 없을 때
    if not video_ids:
        print('새로 수집할 video_id가 없습니다.')
        return pd.DataFrame()
    
    for i in range(0, len(video_ids), 50):
        batch_ids = video_ids[i:i+50]
        
        params = {
            'part': 'snippet,statistics,contentDetails,status,paidProductPlacementDetails',
            'id': ','.join(batch_ids),
            'key': api_key
        }
    
        r = requests.get(url, params=params, timeout=30)
    
        if not r.ok:
            print('status_code :', r.status_code)
            try:
                print('error_json :', r.json())
            except Exception:
                print('error_text :', r.text)
            break
        
        data = r.json()
        items = data.get('items', [])
        
        for item in items:
            snippet = item.get('snippet', {})
            statistics = item.get('statistics',{})
            content_details = item.get('contentDetails', {})
            status = item.get('status', {})
            ppl = item.get('paidProductPlacementDetails', {})
            # file_details = item.get('fileDetails', {})
            # processing_details = item.get('processingDetails',{})
            # suggestions = item.get('suggestions', {})
            
            all_results.append({
                'video_id' : item.get('id'), # 영상 아이디
                'title' : snippet.get('title'), # 영상 제목
                'channel_id' : snippet.get('channelId'), # 채널 아이디
                'channel_title' : snippet.get('channelTitle'), # 채널 이름
                'description' : snippet.get('description'), # 영상 설명
                'upload_date' : snippet.get('publishedAt'), # 영상 업로드 날짜
                'tags' : ', '.join(snippet.get('tags', [])), # 영상 키워드 태그 목록
                'category_id' : snippet.get('categoryId'), # 영상의 카테고리 아이디
                'view_count' : statistics.get('viewCount'), # 영상의 조회수(Shorts 영상은 최소 시청조건 없음. 재생 또는 다시보기 시작되는 횟수 반환)
                'like_count' : statistics.get('likeCount'), # 영상에 좋아요를 표시한 사용자 수
                'comment_count' : statistics.get('commentCount'), # 영상의 댓글 개수
                'duration' : content_details.get('duration'), # 영상 길이 (PT#H#M#S 형식)
                'definition' : content_details.get('definition'), # 영상을 고화질로 시청할 수 있는지 여부 (hd, sd 형식)
                # 'content_rating' : content_details.get('contentRating'), # 다양한 평가 기준에 따라 동영상이 받은 평가
                # 'kmrb_rating' : content_details.get('contentRating', {}).get('kmrbRating'), # 대한민국 영상물등급위원회(KMRB) 등급(예: 전체 관람가)
                'license' : status.get('license'), # 동영상 라이선스 (creativeCommon: 타인의 영상 재편집, 재배포 가능, 출처 표기 / youtube: 재사용/수정 불가)
                'embeddable' : status.get('embeddable'), # 영상을 다른 웹사이트에 퍼갈 수 있는가?
                'has_paid_product_placement' : ppl.get('hasPaidProductPlacement'), # 유료 PPL이 사용되는 경우 true, 아니면 False
                # 'width_pixels' : file_details.get('videoStream[]', []).get('widthPixels'), # 콘텐츠 가로 너비
                # 'height_pixels' : file_details.get('videoStream[]', []).get('heightPixels'), # 콘텐츠 세로 높이
                # 'aspect_ratio' : file_details.get('videoStream[]', []).get('aspectRatio'), # 콘텐츠 가로세로 비율
                # 'vendor' : file_details.get('videoStream[]', []).get('vendor'), # 동영상 공급업체 코드
                # 'tag_suggestions_availability' : processing_details.get('tagSuggestionsAvailability'), # 동영상에 키워드 추천을 사용할 수 있는지 여부 (동영상을 더 쉽게 찾을 수 있도록 동영상의 메타데이터에 태그 추가 가능.)
                # 'tag' : suggestions.get('tagSuggestions[]', []).get('tag'), # 영상에 추천된 키워드 태그 
                # 'category_restiricts' : suggestions.get('tagSuggestions[]', []).get('categoryRestricts[]'), # 태그와 관련된 동영상 카테고리 집합
                # 'caption' : content_details.get('caption'), # 자막을 사용할 수 있는지 여부(bool)
                # 'thumbnail' : snippet.get('thumbnails', {}).get('high',{}).get('url'), # 영상 썸네일(고해상도)
            })
        
    # 영상정보 결과 데이터 프레임으로 저장
    df= pd.DataFrame(all_results)
    
    # 파일로 저장.
    if save_path and not df.empty: # 경로가 존재하고, 데이터 프레임이 비어있지 않다면,
        os.makedirs(os.path.dirname(save_path), exist_ok=True) if os.path.dirname(save_path) else None
        
        if not os.path.exists(save_path):
            df.to_csv(save_path, index=False, encoding='utf-8-sig')
        else:
            df.to_csv(save_path, mode='a', header=False, index=False, encoding='utf-8-sig')
    return df

In [ ]:
# =======================================================================
# 유닛 비용 산출 및 가져올 영상 수 정의 : main_video_info(arg1, arg2, arg3, arg4, arg5)
# 
# arg1: 유튜브 API 키
# arg2: 유튜브 API 키 (예비1)
# arg3: 유튜브 API 키 (예비2)
# arg4: 영상 아이디 리스트
# arg5: csv 파일 저장 경로 (예: save_path = './data/result/data1.csv')
#
# [설명]
# 가진 유닛에 맞게 비용을 산출하여 영상 상세정보를 가져오는 함수로 상세정보를 가져오는 함수입니다.
# 이 함수는 이미 해당경로에 불러온 영상 상세정보를 필터링하여 중복을 방지하는 기능도 포함하고 있습니다.
# 남은 유닛 수를 자동으로 가져올 수 없으므로 수동으로 체크해서 가져오셔야 합니다.
# =======================================================================
def main_video_info(youtube_api_key=None, youtube_api_key2=None, youtube_api_key3=None, video_ids=None, save_path=None):
        
    if save_path is None:
        raise ValueError('파일을 저장할 경로를 입력해주세요')
    
    if video_ids is None:
        raise ValueError('영상 아이디 리스트를 입력해주세요.')
    
    # 사용 가능한 유닛 수 계산
    print('='*25, '시작', '='*25)
    print('남은 유닛 수 확인 : https://console.cloud.google.com/iam-admin/quotas?hl=ko')
    
    unit1 = unit2 = unit3 = 0
    if youtube_api_key is not None:
        unit1 = 10000 - int(input('첫 번째 API의 사용한 유닛 수를 입력해주세요(사용량 입력): '))
    if youtube_api_key2 is not None:
        unit2 = 10000 - int(input('두 번째 API의 사용한 유닛 수를 입력해주세요(사용량 입력): '))
    if youtube_api_key3 is not None:
        unit3 = 10000 - int(input('세 번째 API의 사용한 유닛 수를 입력해주세요(사용량 입력): '))
    
    if (youtube_api_key is None) and (youtube_api_key2 is None) and (youtube_api_key3 is None):
        raise ValueError('유튜브 API 키를 입력해주세요.')
    
    total_units = unit1 + unit2 + unit3
    if total_units == 0:
        print('사용 가능한 유닛이 없습니다.')
        return
    
    for api_key in [youtube_api_key, youtube_api_key2, youtube_api_key3]:
        # api_key 있는 것만 사용
        if api_key is None:
            continue
        # 이미 저장 완료된 video_id 가져오기
        existing_video_ids = set()
        if save_path and os.path.exists(save_path):
            existing_df = pd.read_csv(save_path)
            if 'video_id' in existing_df.columns:
                existing_video_ids = set(existing_df['video_id'].dropna().astype(str)) # video_id 가져오기 완료
    
        # 중복 제거된 video_ids 다시 산출
        video_ids = [x for x in video_ids if x not in existing_video_ids]
        
        if len(video_ids) == 0:
            break
        # 조회 가능한 데이터 수
        print('가지고 있는 유닛 수: ', total_units)
        print('총 video_id 개수: ', len(video_ids))
        possible = total_units * 50
        
        print(f'{len(video_ids)}개의 영상 중 {possible}개의 영상이 조회가 가능합니다.')
        get_videos = 0
        while total_units > 0:
            # 조회 가능한 영상 수인지 확인하기
            get_videos = input(f'{len(video_ids)}개의 영상 중 몇 개의 영상을 조회하시겠습니까? (종료: N) :')
            if get_videos == 'N':
                break
            else:
                get_videos =int(get_videos)
                
            if np.ceil(get_videos / 50) > total_units:
                print('가지고 있는 유닛 수로 조회하기엔 영상이 너무 많습니다.')
                get_videos = 0
            
            if total_units == 0:
                break
            
            if get_videos == 0:
                break
            print(f'예상 비용 : {np.ceil(get_videos/50)} 유닛')
            video_list = video_ids[0:get_videos]
            
            get_video_info(api_key=api_key, video_ids=video_list, save_path=save_path)
            total_units = total_units - (np.ceil(get_videos/50))
            
            video_ids = [x for x in video_ids if x not in video_list]
            
            print(f'남은 유닛: {total_units}')
            if total_units <= 0:
                break
        if total_units <= 0:
            break
    return print(f'추출이 완료되었습니다. 파일 경로: {save_path}')


In [ ]:
# =======================================================================
# IT 영상 상세정보 남은 토큰 수에 맞게 가져오기
# =======================================================================
it_videos = pd.read_csv('./data/IT_video_ids.csv')
video_ids = it_videos['video_id'].to_list()
main_video_info(youtube_api_key=youtube_api_key, youtube_api_key2=youtube_api_key2, youtube_api_key3=youtube_api_key3, video_ids=video_ids, save_path='./data/results/it_video_info.csv')




In [ ]:
# =======================================================================
# FOOD 영상 상세정보 남은 토큰 수에 맞게 가져오기
# =======================================================================
food_videos = pd.read_csv('./data/FOOD_video_ids.csv')
video_ids = food_videos['video_id'].to_list()
main_video_info(youtube_api_key=youtube_api_key, youtube_api_key2=youtube_api_key2, youtube_api_key3=youtube_api_key3, video_ids=video_ids, save_path='./data/results/food_video_info.csv')



In [2]:
# 저장된 IT 채널의 영상 상세정보 파일 가져오기
it_video_info = pd.read_csv('./data/results/it_video_info.csv')
print(len(it_video_info))

# 저장된 F&B 채널의 영상 상세정보 파일 가져오기
food_video_info = pd.read_csv('./data/results/food_video_info.csv')
print(len(food_video_info))

20435
21248


---
### 롱폼/숏폼 분류
- 롱폼과 숏폼을 구분해서 따로 분석할 계획
---

In [30]:
# =======================================================================
# 롱폼/숏폼 분류기 : who_long_short_form(arg1, arg2, arg3)
# arg1: DataFrame 변수 이름입니다. (무조건 데이터 프레임 형태)
# arg2: 파일을 저장할 경로입니다! (예: save_path = './data/example.csv')
# arg3: 일꾼의 수 입니다. (기본값: 20)
#
# [작동 원리]
# - Request를 이용하여 youtube/shorts/video_id 형태의 url을 입력하여 HTTP로 호출합니다.
# - 해당 url의 영상이 롱폼이라면 youtube/watch/v=video_id 형태로 변환되어 접속됩니다.
# - response.url을 이용하여 최종적으로 어떤 형식의 url으로 접속되었는지에 따라 롱폼/숏폼으로 구분합니다.
#
# 
# =======================================================================


def who_long_short_form(data=None, save_path=None, workers=20):
    # 1. 데이터 검증 및 준비
    if data is None or not isinstance(data, pd.DataFrame):
        raise ValueError('DataFrame의 이름을 입력해주세요')
    
    if 'video_id' not in data.columns:
        raise KeyError("해당 데이터프레임에 'video_id' 컬럼이 없습니다.")
    
    # 중복 제거된 비디오 ID 리스트 (속도 향상을 위해 고유값만 추출)
    video_ids = data['video_id'].dropna().unique().tolist()
    total_ids = len(video_ids)
    
    # --- 설정값 및 헤더 ---
    HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"}
    
    print(f"🚀 총 {total_ids}개 영상 병렬 분석 시작 (Workers: {workers})...\n")

    # [내부 함수] 개별 영상 분류 로직
    def classify_video(vid):
        url = f'https://www.youtube.com/shorts/{vid}'
        try:
            # allow_redirects=True로 설정하여 최종 URL 확인
            response = requests.get(url, timeout=10, headers=HEADERS)
            final_url = response.url
            raw_html = response.text.replace(" ", "")
            
            # 재생 불가 상태 체크
            if '"playabilityStatus":{"status":"ERROR"' in raw_html or '"playabilityStatus":{"status":"UNPLAYABLE"' in raw_html:
                verdict = 'error'
            elif '/watch' in final_url:
                verdict = 'longform'
            elif '/shorts' in final_url:
                verdict = 'shorts'
            else:
                verdict = 'unknown'
                
            return {'video_id': vid, 'verdict': verdict, 'final_url': final_url}
        
        except Exception as e:
            return {'video_id': vid, 'verdict': 'error', 'final_url': f'Error: {str(e)}'}

    # 2. 멀티쓰레딩 실행
    results = []
    with ThreadPoolExecutor(max_workers=workers) as executor:
        # 진행률 표시를 위해 as_completed 사용
        futures = {executor.submit(classify_video, vid): vid for vid in video_ids}
        
        for i, future in enumerate(as_completed(futures), 1):
            res = future.result()
            results.append(res)
            
            # IPYNB에서 너무 많은 출력을 방지하기 위해 50개마다 진행 상황 출력
            if i % 50 == 0 or i == total_ids:
                print(f"✅ 진행 상황: [{i}/{total_ids}] 완료")

    # 3. 결과 합치기 및 저장
    new_df = pd.DataFrame(results)
    merge_df = pd.merge(data, new_df, on='video_id', how='left')
    
    if save_path:
        os.makedirs(os.path.dirname(save_path) if os.path.dirname(save_path) else '.', exist_ok=True)
        merge_df.to_csv(save_path, index=False, encoding='utf-8-sig')
        print(f"\n💾 결과 저장 완료: {save_path}")
        
    return merge_df

# --- 사용 예시 ---
# food_video_info = pd.read_csv("your_data.csv")
# result_df = who_long_short_fast(food_video_info, save_path='./data/results/food_redirect.csv', workers=30)
# display(result_df.head())


In [ ]:
food_redirect = who_long_short_form(food_video_info, save_path='./data/results/food_redirect.csv', workers=30)
display(food_redirect.head())

🚀 총 21248개 영상 병렬 분석 시작 (Workers: 30)...

✅ 진행 상황: [50/21248] 완료
✅ 진행 상황: [100/21248] 완료
✅ 진행 상황: [150/21248] 완료
✅ 진행 상황: [200/21248] 완료
✅ 진행 상황: [250/21248] 완료
✅ 진행 상황: [300/21248] 완료
✅ 진행 상황: [350/21248] 완료
✅ 진행 상황: [400/21248] 완료
✅ 진행 상황: [450/21248] 완료
✅ 진행 상황: [500/21248] 완료
✅ 진행 상황: [550/21248] 완료
✅ 진행 상황: [600/21248] 완료
✅ 진행 상황: [650/21248] 완료
✅ 진행 상황: [700/21248] 완료
✅ 진행 상황: [750/21248] 완료
✅ 진행 상황: [800/21248] 완료
✅ 진행 상황: [850/21248] 완료
✅ 진행 상황: [900/21248] 완료
✅ 진행 상황: [950/21248] 완료
✅ 진행 상황: [1000/21248] 완료
✅ 진행 상황: [1050/21248] 완료
✅ 진행 상황: [1100/21248] 완료
✅ 진행 상황: [1150/21248] 완료
✅ 진행 상황: [1200/21248] 완료
✅ 진행 상황: [1250/21248] 완료
✅ 진행 상황: [1300/21248] 완료
✅ 진행 상황: [1350/21248] 완료
✅ 진행 상황: [1400/21248] 완료
✅ 진행 상황: [1450/21248] 완료
✅ 진행 상황: [1500/21248] 완료
✅ 진행 상황: [1550/21248] 완료
✅ 진행 상황: [1600/21248] 완료
✅ 진행 상황: [1650/21248] 완료
✅ 진행 상황: [1700/21248] 완료
✅ 진행 상황: [1750/21248] 완료
✅ 진행 상황: [1800/21248] 완료
✅ 진행 상황: [1850/21248] 완료
✅ 진행 상황: [1900/21248] 완료
✅ 진행 상황: [1950/21248] 완료
✅ 진행

,video_id,title,channel_id,channel_title,description,upload_date,tags,category_id,view_count,like_count,comment_count,duration,definition,license,embeddable,has_paid_product_placement,thumbnail,thumbnail_h,thumbnail_w,caption,verdict,final_url
0,gVdoQQI53HU,우베가 CU에도 나왔나배 | [톡! 까놓고 신상리뷰💬],UCsM07dUwo0WOWhFbVsG8K6A,CU [씨유튜브],우베 디저트부터\n베이크하우스 405 담당 MD가 \n직접 소개하는 빵 신상까지!\...,2026-04-20T09:30:31Z,"CU편의점, CU, 씨유, 헤이루, Heyroo",20,2228,49.0,5.0,PT17M56S,hd,youtube,True,False,https://i.ytimg.com/vi/gVdoQQI53HU/hqdefault.jpg,360,480,False,longform,https://www.youtube.com/watch?v=gVdoQQI53HU
1,FuZKyvWuWP8,데우면 더 위험해지는 초코바게트 [톡! 까놓고 신상리뷰],UCsM07dUwo0WOWhFbVsG8K6A,CU [씨유튜브],"초코칩과 가나슈가 듬뿍! 유행 초코템🍫\n📌405 초코바게트 3,200원\n\n지금...",2026-04-17T09:30:18Z,"CU편의점, CU, 씨유, 헤이루, Heyroo",20,144318,272.0,0.0,PT16S,hd,youtube,True,False,https://i.ytimg.com/vi/FuZKyvWuWP8/hqdefault.jpg,360,480,False,shorts,https://www.youtube.com/shorts/FuZKyvWuWP8
2,_1T0CAWGsPE,두바이 버터떡은 무슨 맛일까? [톡! 까놓고 신상리뷰],UCsM07dUwo0WOWhFbVsG8K6A,CU [씨유튜브],"푸짐한 카다이프 페이스트를 쫄깃하게 감싼 버터떡🫶\n📌조이 두바이버터떡 3,900원...",2026-04-16T10:01:04Z,"CU편의점, CU, 씨유, 헤이루, Heyroo",20,135901,242.0,2.0,PT18S,hd,youtube,True,False,https://i.ytimg.com/vi/_1T0CAWGsPE/hqdefault.jpg,360,480,False,shorts,https://www.youtube.com/shorts/_1T0CAWGsPE
3,zrhBNe2EMNY,CU가 유행 끝내겠습니다 | [톡! 까놓고 신상리뷰💬],UCsM07dUwo0WOWhFbVsG8K6A,CU [씨유튜브],두쫀쿠를 품은 버터떡에😋\n생과일을 품은 프리미엄 하이볼까지!🍊\n\n4월 2주차 ...,2026-04-13T10:01:41Z,"CU편의점, CU, 씨유, 헤이루, Heyroo",20,105129,86.0,7.0,PT16M43S,hd,youtube,True,False,https://i.ytimg.com/vi/zrhBNe2EMNY/hqdefault.jpg,360,480,False,longform,https://www.youtube.com/watch?v=zrhBNe2EMNY
4,kgajE44zZfo,짭짤하게 즐기는 황치즈쫀득쿠키 [톡! 까놓고 신상리뷰],UCsM07dUwo0WOWhFbVsG8K6A,CU [씨유튜브],"황치즈 러버라면 지나칠 수 없는 디저트 출시!\n📌황치즈쫀득쿠키 3,500원\n\n...",2026-04-10T09:01:16Z,"CU편의점, CU, 씨유, 헤이루, Heyroo",20,100489,185.0,0.0,PT26S,hd,youtube,True,False,https://i.ytimg.com/vi/kgajE44zZfo/hqdefault.jpg,360,480,False,shorts,https://www.youtube.com/shorts/kgajE44zZfo


In [31]:
it_redirect = who_long_short_form(it_video_info, save_path='./data/results/it_redirect.csv', workers=30)
display(it_redirect.head())

🚀 총 20435개 영상 병렬 분석 시작 (Workers: 30)...

✅ 진행 상황: [50/20435] 완료
✅ 진행 상황: [100/20435] 완료
✅ 진행 상황: [150/20435] 완료
✅ 진행 상황: [200/20435] 완료
✅ 진행 상황: [250/20435] 완료
✅ 진행 상황: [300/20435] 완료
✅ 진행 상황: [350/20435] 완료
✅ 진행 상황: [400/20435] 완료
✅ 진행 상황: [450/20435] 완료
✅ 진행 상황: [500/20435] 완료
✅ 진행 상황: [550/20435] 완료
✅ 진행 상황: [600/20435] 완료
✅ 진행 상황: [650/20435] 완료
✅ 진행 상황: [700/20435] 완료
✅ 진행 상황: [750/20435] 완료
✅ 진행 상황: [800/20435] 완료
✅ 진행 상황: [850/20435] 완료
✅ 진행 상황: [900/20435] 완료
✅ 진행 상황: [950/20435] 완료
✅ 진행 상황: [1000/20435] 완료
✅ 진행 상황: [1050/20435] 완료
✅ 진행 상황: [1100/20435] 완료
✅ 진행 상황: [1150/20435] 완료
✅ 진행 상황: [1200/20435] 완료
✅ 진행 상황: [1250/20435] 완료
✅ 진행 상황: [1300/20435] 완료
✅ 진행 상황: [1350/20435] 완료
✅ 진행 상황: [1400/20435] 완료
✅ 진행 상황: [1450/20435] 완료
✅ 진행 상황: [1500/20435] 완료
✅ 진행 상황: [1550/20435] 완료
✅ 진행 상황: [1600/20435] 완료
✅ 진행 상황: [1650/20435] 완료
✅ 진행 상황: [1700/20435] 완료
✅ 진행 상황: [1750/20435] 완료
✅ 진행 상황: [1800/20435] 완료
✅ 진행 상황: [1850/20435] 완료
✅ 진행 상황: [1900/20435] 완료
✅ 진행 상황: [1950/20435] 완료
✅ 진행

,video_id,title,channel_id,channel_title,description,upload_date,tags,category_id,view_count,like_count,comment_count,duration,definition,license,embeddable,has_paid_product_placement,thumbnail,thumbnail_h,thumbnail_w,caption,verdict,final_url
0,YkE11u3wqcY,[#꿀팁줍숏] 퇴근 1시간 앞당기는 엑셀 단축키 2) Multi Key 📊,UCVaIGU8ch9zG42j-rAG6VoA,삼성SDS,엑셀. 언제까지 마우스로 딸깍거릴 거예요? 실전 단축키 모음 2탄! ⌨\n\nCtr...,2026-04-21T10:00:48Z,"삼성SDS, 삼성에스디에스, 꿀팁줍숏, AI활용법, 생성형AI, FabriX, Br...",28,1004,10.0,0.0,PT1M6S,hd,youtube,True,False,https://i.ytimg.com/vi/YkE11u3wqcY/hqdefault.jpg,360,480,False,shorts,https://www.youtube.com/shorts/YkE11u3wqcY
1,NSxpdlnn9wo,[#꿀팁줍숏] 퇴근 1시간 앞당기는 엑셀 단축키 1) Single Key 📊,UCVaIGU8ch9zG42j-rAG6VoA,삼성SDS,엑셀. 아직도 마우스로 하나하나 작업하며 시간 낭비 중?\n그거 계속하면 손목 터널...,2026-04-14T10:00:28Z,"삼성SDS, 삼성에스디에스, 꿀팁줍숏, AI활용법, 생성형AI, FabriX, Br...",28,2623,42.0,1.0,PT1M3S,hd,youtube,True,False,https://i.ytimg.com/vi/NSxpdlnn9wo/hqdefault.jpg,360,480,False,shorts,https://www.youtube.com/shorts/NSxpdlnn9wo
2,0mRKMcTFPAY,[#꿀팁줍숏] 일은 잘 시켜야 잘한다! 생성형 AI 프롬프트 잘 뽑는 법 ✨,UCVaIGU8ch9zG42j-rAG6VoA,삼성SDS,"너도 나도 다 쓰는 생성형 AI, 업무 중 쓰려니 자꾸 엉뚱한 대답만 내놓는다고? ...",2026-04-08T00:53:48Z,"삼성SDS, 삼성에스디에스, 꿀팁줍숏, 프롬프트, AI활용법, 생성형AI, Fabr...",28,716,10.0,0.0,PT1M3S,hd,youtube,True,False,https://i.ytimg.com/vi/0mRKMcTFPAY/hqdefault.jpg,360,480,False,shorts,https://www.youtube.com/shorts/0mRKMcTFPAY
3,mBu4qOdH8aI,[웨비나] 지능형 AI 협업으로 시작하는 공공부문 일하는 방식의 변화,UCVaIGU8ch9zG42j-rAG6VoA,삼성SDS,"반복되는 문서 작성, 부서 간 복잡한 협업..\n“조금 더 효율적인 방법은 없을까?...",2026-04-06T09:23:18Z,NaN,28,830,14.0,1.0,PT1H2M35S,hd,youtube,True,False,https://i.ytimg.com/vi/mBu4qOdH8aI/hqdefault.jpg,360,480,False,longform,https://www.youtube.com/watch?v=mBu4qOdH8aI
4,6D2ueCdbbes,"봄, 사랑, 벚꽃 말고 이거 챙겨라~! 🌸 feat. 2026년 4월 석촌호수 개발...",UCVaIGU8ch9zG42j-rAG6VoA,삼성SDS,벚꽃하면 어디? 석촌호수 🌸\n미신이요? 하면 지평좌표계 먼저 찾는 슈퍼T 부기 앞...,2026-04-02T10:00:09Z,"석촌호수, 석촌호수 실시간, 석촌호수 벚꽃축제, 벚꽃축제, 잠실 벚꽃, 서울 벚꽃,...",28,341,6.0,0.0,PT49S,hd,youtube,True,False,https://i.ytimg.com/vi/6D2ueCdbbes/hqdefault.jpg,360,480,False,longform,https://www.youtube.com/watch?v=6D2ueCdbbes


In [34]:
it_redirect[it_redirect['verdict']=='error']

,video_id,title,channel_id,channel_title,description,upload_date,tags,category_id,view_count,like_count,comment_count,duration,definition,license,embeddable,has_paid_product_placement,thumbnail,thumbnail_h,thumbnail_w,caption,verdict,final_url
4967,GALDe3pyq48,"[B tv 영화 추천/movie Big #20] 오션스 8, 도둑들, 섹션제로3: ...",UCwR9k7QggFEfeHkvTR31JcA,SK브로드밴드_B tv,[금주의 빅무비] - 섹션제로3: 블록13\n[무비빅스타] - 영화배우 김환희\n[...,2018-08-03T02:54:09Z,"SK브로드밴드, SKB, B tv, movie Big, 백업무비, 버킷무비, 무비빅...",1,404,1.0,0.0,PT54M18S,hd,youtube,True,False,https://i.ytimg.com/vi/GALDe3pyq48/hqdefault.jpg,360,480,False,error,https://www.youtube.com/watch?v=GALDe3pyq48
4974,3tzvP_ZPReg,[B tv 영화 추천/movie Big #20] 오션스 8 vs 도둑들,UCwR9k7QggFEfeHkvTR31JcA,SK브로드밴드_B tv,"[B tv 영화 추천/movie Big #20] 오션스 8(Ocean's 8 , 2...",2018-07-26T09:14:19Z,"SK브로드밴드, SKB, B tv, movie Big, 백업무비, 버킷무비, 무비빅...",1,376,1.0,0.0,PT17M27S,hd,youtube,True,False,https://i.ytimg.com/vi/3tzvP_ZPReg/hqdefault.jpg,360,480,False,error,https://www.youtube.com/watch?v=3tzvP_ZPReg
5300,sBM4pxyiXg4,"[B tv 영화 추천] 아토믹 블론드 (Atomic Blonde , 2017)",UCwR9k7QggFEfeHkvTR31JcA,SK브로드밴드_B tv,[영화로운 세상] 웰컴 투 무비 - 아토믹 블론드,2017-10-25T07:00:02Z,"SK B tv, B tv, 비티비, 아토믹 블론드, Atomic Blonde, 영화...",1,544,3.0,0.0,PT3M51S,hd,youtube,True,False,https://i.ytimg.com/vi/sBM4pxyiXg4/hqdefault.jpg,360,480,False,error,https://www.youtube.com/watch?v=sBM4pxyiXg4
5308,oUAP9Z8X_Ew,"[김태훈의 무비셀렉션] 윈드 리버 (Wind River , 2016)",UCwR9k7QggFEfeHkvTR31JcA,SK브로드밴드_B tv,[영화로운 세상] 무비 셀렉션 - 윈드 리버,2017-10-22T04:00:01Z,"SK B tv, B tv, 비티비, 윈드 리버, Wind River, 영화추천, 추...",1,2092,15.0,0.0,PT8M7S,hd,youtube,True,False,https://i.ytimg.com/vi/oUAP9Z8X_Ew/hqdefault.jpg,360,480,False,error,https://www.youtube.com/watch?v=oUAP9Z8X_Ew
5527,qchTQoMeGME,"[B tv 영화 추천] 재심 (New Trial, 2016)",UCwR9k7QggFEfeHkvTR31JcA,SK브로드밴드_B tv,[영화로운 세상] 지혜로운 태용씨 - 재심,2017-04-13T03:27:20Z,"SK B tv, B tv, 비티비, 재심, 정우, 강하늘, 김해숙, 이동휘, 한재영...",1,521,5.0,0.0,PT12M10S,hd,youtube,True,False,https://i.ytimg.com/vi/qchTQoMeGME/hqdefault.jpg,360,480,False,error,https://www.youtube.com/watch?v=qchTQoMeGME
5879,Emq8naf4RlM,이번 주 뭘 볼까? 영화 '님아 그강을 건너지 마오',UCwR9k7QggFEfeHkvTR31JcA,SK브로드밴드_B tv,"우리는 76년째 연인입니다.\n89세 소녀감성 강계열 할머니, 98세 로맨티스트 조...",2015-02-13T10:01:08Z,"SK B tv, B tv, 비티비, 영화 추천, 님아 그강을 건너지 마오",24,6445,15.0,0.0,PT7M49S,hd,youtube,True,False,https://i.ytimg.com/vi/Emq8naf4RlM/hqdefault.jpg,360,480,True,error,https://www.youtube.com/watch?v=Emq8naf4RlM
7464,twRn4exv0Z8,if(kakao)2020 / 비즈니스. 톡처럼 쉬워지다.,UCBjvBJgIp3NGkrTBEfWBUVw,카카오,https://if.kakao.com\n카카오가 여러분의 비즈니스를 쉽게 성장시킬 ...,2020-11-19T04:00:05Z,NaN,22,1757,16.0,1.0,PT16M1S,hd,youtube,True,False,https://i.ytimg.com/vi/twRn4exv0Z8/hqdefault.jpg,360,480,False,error,https://www.youtube.com/watch?v=twRn4exv0Z8&pp...
15474,KkcKEYLIblw,"밍찌니, 문도 피구 판 떠나라",UCMaDyFlagYlsolLLfWHu7Dw,숲 (SOOP),밍찌니♥ : https://bj.afreecatv.com/mjinhee\n\n아프리...,2021-11-02T09:00:52Z,"아프리카TV, AfreecaTV, BJ, 비제이, 먹방, Mukbang, 생방송, ...",20,842,NaN,0.0,PT2M28S,hd,youtube,True,False,https://i.ytimg.com/vi/KkcKEYLIblw/hqdefault.jpg,360,480,False,error,https://www.youtube.com/watch?v=KkcKEYLIblw
17243,L_DiGKiEfiw,"리니지2 레볼루션, 수요공썰전 시즌5 2회, 난닝구출연 #1",UCMaDyFlagYlsolLLfWHu7Dw,숲 (SOOP),아프리카TV - http://afreecatv.com\nYouTube - https...,2018-02-01T06:57:24Z,"아프리카, 아프리카TV, AfreecaTV, BJ, 비제이, 생방, 녹방, 게임, ...",20,300,NaN,0.0,PT1H2S,hd,youtube,True,False,https://i.ytimg.com/vi/L_DiGKiEfiw/hqdefault.jpg,360,480,False,error,https://www.youtube.com/watch?v=L_DiGKiEfiw
17258,CnslQ6NuQpo,[곽방TV]최초의 반려동물은 어떻게 탄생했을까? - 길들이기 특집 #1,UCMaDyFlagYlsolLLfWHu7Dw,숲 (SOOP),아프리카TV - http://afreecatv.com\nYouTube - https...,2018-01-30T09:37:05Z,"아프리카, 아프리카TV, AfreecaTV, BJ, 비제이, 생방, 녹방, 게임, ...",20,242,NaN,0.0,PT1H2S,hd,youtube,True,False,https://i.ytimg.com/vi/CnslQ6NuQpo/hqdefault.jpg,360,480,False,error,https://www.youtube.com/watch?v=CnslQ6NuQpo


---
### 2차 필터링
- 
---

In [53]:
food_redirect = food_redirect[food_redirect['upload_date'] >= '2021-07-13']

In [57]:
counts = food_redirect.groupby('channel_id')['verdict'].value_counts()

valid_channels = counts[counts >= 100]

valid_channels
# reset_index()['channel_id'].value_counts()

channel_id                verdict 
UC-9HdRV-OzriZ1Q09bT304w  longform    136
                          shorts      106
UC-WFvgdXXmZPj4pDI8GFbzQ  shorts      165
                          longform    101
UCC3eqZIo6lqZQQAN9MO0LQA  shorts      414
                          longform    320
UCDjdk0ufvcvV08hAm_vF06Q  longform    122
UCDryA2PIGnbs9NYn_63rISw  shorts      147
UCEKRI0fipK4LEgV98nI2CQA  longform    117
UCFXwXfbr4b1HHl-YKCOaydA  longform    297
                          shorts      251
UCG_gfzGVpsxGwYuFgH8-tNA  longform    130
UCGgQBs1IgNkwMYbnu26S1yg  longform    183
                          shorts      129
UCGmLtNZFTL6PiKsz3Vpdgbw  longform    136
UCHvCHuXTg6yfQQ1QCeDQMLg  shorts      256
                          longform    130
UCOV_BpGzvmH7ZtjQOWg0bSQ  longform    125
UCSQ0lVqv57JVw1lOdJ2vTVA  shorts      476
                          longform    130
UCUQhyOLi8uFTmiFRmndjQxw  longform    156
UCaxvyTYdWaDXupmgj5ttDUQ  shorts      459
                          longform    180

In [55]:
it_redirect = it_redirect[it_redirect['upload_date'] >= '2021-07-13']

counts = it_redirect.groupby('channel_id')['verdict'].value_counts()

valid_channels = counts[counts >= 100]

len(valid_channels.reset_index()['channel_id'].value_counts())

12

In [ ]:
# =======================================================================
# 채널 2차 필터링
# 1. 쇼츠 서비스 시작일 이전 데이터 제거
# 2. 롱폼, 쇼츠 개수 각 00개씩 제한
#  
# =======================================================================

def channel_second_filtering(data):
    # ==============================================
    # 1. 쇼츠 서비스 시작일 이전 데이터 제거
    # ==============================================
    data = data[data['upload_date'] >= '2021-07-13']
    
    # ==============================================
    # 2. 롱폼 00개 미만, 숏폼 00개 미만인 채널 제거
    # ==============================================
    # 
    counts = data['verdict'].value_counts()
    

In [ ]:
food_video_info['upload_date'] = pd.to_datetime(food_video_info['upload_date'], errors='coerce', format='mixed')
it_video_info['upload_date'] = pd.to_datetime(it_video_info['upload_date'], errors='coerce', format='mixed')